# 1장. 다음 토큰 예측을 숫자로 재기

작은 트랜스포머 언어 모델을 더 오래 학습시키고, 손실을 다른 단위로 바꾸고, 검증 자료와 만든 문장을 다시 잰다.

**준비:** NSMC의 `ratings_train.txt`를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. GPU가 없어도 된다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 24분쯤 걸렸다.

## 1.1 출발점: 작은 트랜스포머와 3-그램

In [ ]:
import math, time, json, sys, platform
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

torch.set_num_threads(2)                            # 코랩 CPU와 같은 조건
raw = pd.read_csv("ratings_train.txt", sep="\t").dropna(subset=["document"])
raw = raw.reset_index(drop=True)
order = np.random.default_rng(0).permutation(len(raw))   # 1부 내내 같은 순서로 나눈다
train, valid = raw.iloc[order[:30000]], raw.iloc[order[-10000:]]
spare = raw.iloc[order[30000:-10000]]
texts_tr, texts_va = train["document"].tolist(), valid["document"].tolist()
texts_un = spare["document"].tolist()        # 언어 모델은 이 문장으로 학습한다

PAD, UNKNOWN = 0, 1
counts = Counter(ch for text in texts_tr for ch in text)
vocab = {ch: k + 2 for k, (ch, n) in enumerate(counts.most_common()) if n >= 5}
V = len(vocab) + 2
START, END = V, V + 1

def encode(texts, length=64):
    """<시작> + 글자 번호 + <끝>. length + 2칸이고 남는 칸은 0이다."""
    ids = torch.zeros(len(texts), length + 2, dtype=torch.long)
    for row, text in enumerate(texts):
        found = [START] + [vocab.get(ch, UNKNOWN) for ch in text][:length] + [END]
        ids[row, :len(found)] = torch.tensor(found, dtype=torch.long)
    return ids

X_tr, X_va, X_un = encode(texts_tr), encode(texts_va), encode(texts_un)
print(len(texts_un), len(texts_va), V + 2, X_un.shape)
print(sys.version.split()[0], np.__version__, torch.__version__,   # 환경을 남긴다
      platform.machine(), torch.get_num_threads())

In [ ]:
class SelfAttention(nn.Module):
    """셀프 어텐션(인과 마스크를 건다). x: (배치, 길이, dim)."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.to_q, self.to_k, self.to_v = (nn.Linear(dim, dim) for _ in range(3))
        self.out = nn.Linear(dim, dim)

    def forward(self, x, hide):
        B, L, D = x.shape
        H = self.heads
        split = lambda t: t.view(B, L, H, D // H).transpose(1, 2)    # (B, H, L, D/H)
        q, k, v = split(self.to_q(x)), split(self.to_k(x)), split(self.to_v(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(D // H)
        scores = scores.masked_fill(hide[:, None, None, :], float("-inf"))
        later = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(later, float("-inf"))           # 인과 마스크
        mixed = torch.softmax(scores, dim=-1) @ v
        return self.out(mixed.transpose(1, 2).reshape(B, L, D))

class Block(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attend = SelfAttention(dim, heads)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(),
                                 nn.Linear(4 * dim, dim))

    def forward(self, x, hide):
        x = x + self.attend(self.norm1(x), hide)
        return x + self.mlp(self.norm2(x))

class LanguageModel(nn.Module):
    """작은 트랜스포머 언어 모델: 임베딩 + 블록 2개 + 층 정규화 + 출력 층."""
    def __init__(self, dim=64, heads=4, layers=2, length=66):
        super().__init__()
        self.table = nn.Embedding(V + 2, dim)
        self.where = nn.Embedding(length, dim)
        self.blocks = nn.ModuleList(Block(dim, heads) for _ in range(layers))
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, V + 2)

    def forward(self, ids):                          # (배치, 길이, V + 2)
        x = self.table(ids) + self.where(torch.arange(ids.shape[1]))
        hide = ids == PAD
        for block in self.blocks:
            x = block(x, hide)
        return self.head(self.norm(x))

count = lambda net: sum(p.numel() for p in net.parameters())
torch.manual_seed(0)
print(count(LanguageModel()))

In [ ]:
def batches(lengths, seed, size=64, chunk=50):
    """길이가 비슷한 문장끼리 배치를 만든다(패딩을 줄여 계산을 아낀다)."""
    shuffler = torch.Generator().manual_seed(seed)
    mixed = torch.randperm(len(lengths), generator=shuffler)
    made = []
    for start in range(0, len(mixed), size * chunk):
        part = mixed[start:start + size * chunk]
        part = part[torch.argsort(lengths[part])]
        made += [part[k:k + size] for k in range(0, len(part), size)]
    return [made[k] for k in torch.randperm(len(made), generator=shuffler)]

def lm_loss(net, ids):
    out = net(ids[:, :-1])
    return F.cross_entropy(out.reshape(-1, out.shape[-1]), ids[:, 1:].reshape(-1),
                           ignore_index=PAD)

def lm_score(net, data):
    """목표 토큰 하나의 평균 손실(자연로그)."""
    net.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for k in range(0, len(data), 500):
            ids = data[k:k + 500]
            out = net(ids[:, :-1])
            total += F.cross_entropy(out.reshape(-1, out.shape[-1]), ids[:, 1:].reshape(-1),
                                     ignore_index=PAD, reduction="sum").item()
            n += (ids[:, 1:] != PAD).sum().item()
    return total / n

def fit(net, X, lr=0.03, seed=0, epochs=2, watch=None):
    """AdamW, 배치 64, 처음 10% 걸음 동안 학습률을 올리고 코사인으로 줄인다.
    watch를 주면 에폭이 끝날 때마다 검증 손실을 적는다."""
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.0)
    lengths = (X != PAD).sum(dim=1)
    total, step = epochs * math.ceil(len(X) / 64), 0
    warm = max(1, total // 10)
    for epoch in range(epochs):
        net.train()
        for rows in batches(lengths, seed * 1000 + epoch):
            step += 1
            wave = 0.5 * (1 + math.cos(math.pi * step / total))
            for group in optimizer.param_groups:
                group["lr"] = lr * min(1, step / warm) * wave
            ids = X[rows][:, :int(lengths[rows].max())]
            optimizer.zero_grad()
            lm_loss(net, ids).backward()
            optimizer.step()
        if watch is not None:
            watch.append(lm_score(net, X_va))
    return net

def align(text, width):
    wide = sum(ord(ch) > 0x1100 for ch in text)
    return text + " " * max(width - len(text) - wide, 1)

In [ ]:
def pieces(texts):
    return [["<시작>"] + list(text)[:64] + ["<끝>"] for text in texts]

class ThreeGram:
    """3-그램: 글자 하나, 앞 글자 하나, 앞 글자 둘 뒤의 횟수를 섞는다."""
    def __init__(self, texts, mix=(0.1, 0.3, 0.6)):
        self.mix = mix
        self.one, self.two, self.three = Counter(), defaultdict(Counter), defaultdict(Counter)
        for line in pieces(texts):
            for k in range(1, len(line)):
                before = line[k - 2] if k >= 2 else "<시작>"
                self.one[line[k]] += 1
                self.two[line[k - 1]][line[k]] += 1
                self.three[before, line[k - 1]][line[k]] += 1
        self.everything = sum(self.one.values())
        self.sums2 = {key: sum(c.values()) for key, c in self.two.items()}
        self.sums3 = {key: sum(c.values()) for key, c in self.three.items()}

    def sentence_loss(self, text):
        """문장 하나의 손실 합(자연로그)과 목표 토큰 수."""
        line, total = pieces([text])[0], 0.0
        for k in range(1, len(line)):
            before = line[k - 2] if k >= 2 else "<시작>"
            p1 = (self.one[line[k]] + 1) / (self.everything + V + 2)
            c2, c3 = self.two.get(line[k - 1]), self.three.get((before, line[k - 1]))
            p2 = c2[line[k]] / self.sums2[line[k - 1]] if c2 else 0.0
            p3 = c3[line[k]] / self.sums3[before, line[k - 1]] if c3 else 0.0
            total -= math.log(self.mix[0] * p1 + self.mix[1] * p2 + self.mix[2] * p3)
        return total, len(line) - 1

    def score(self, texts):
        parts = [self.sentence_loss(t) for t in texts]
        return sum(a for a, _ in parts) / sum(b for _, b in parts)

gram = ThreeGram(texts_un)
GRAM = gram.score(texts_va)
print(round(math.log(V + 2), 4), round(GRAM, 4), round(math.exp(GRAM), 1))

In [ ]:
took = {}
torch.manual_seed(0)
lap = time.perf_counter()
lm2 = fit(LanguageModel(), X_un, seed=0, epochs=2)
took["2에폭"] = time.perf_counter() - lap
OTHER = 3.0079                                        # 같은 코드를 다른 컴퓨터(x86_64)에서 돌린 값
here = lm_score(lm2, X_va)
print(f"{here:.4f} {OTHER} {here - OTHER:+.4f} {GRAM:.4f} {took['2에폭']:.0f}초")

## 1.2 에폭을 늘리며 검증 손실 재기

In [ ]:
RUNS = {2: [0, 1, 2], 4: [0], 8: [0], 16: [0, 1, 2]}   # 에폭 수: 시드
result, curves, kept = {}, {}, {}
for epochs, seeds in RUNS.items():
    for seed in seeds:
        torch.manual_seed(seed)
        lap, watch = time.perf_counter(), []
        net = fit(LanguageModel(), X_un, seed=seed, epochs=epochs, watch=watch)
        result[epochs, seed] = (lm_score(net, X_un[:5000]), watch[-1],
                                time.perf_counter() - lap)
        curves[epochs, seed] = watch
        if seed == 0:
            kept[epochs] = net                       # 1.4절과 1.5절에서 다시 쓴다
print(align("에폭", 6) + "학습 손실  검증 손실 (범위)              시간")
for epochs, seeds in RUNS.items():
    tr = np.median([result[epochs, s][0] for s in seeds])
    va = [result[epochs, s][1] for s in seeds]
    span = f"({min(va):.4f}~{max(va):.4f})" if len(va) > 1 else ""
    sec = np.median([result[epochs, s][2] for s in seeds])
    print(f"{epochs:<6}{tr:.4f}     {np.median(va):.4f} {span:<22}{sec:>5.0f}초")
print([round(v, 4) for v in curves[16, 0]])

In [ ]:
import glob, os
import matplotlib.pyplot as plt
from matplotlib import font_manager
# 한글 글꼴(Noto Sans CJK KR)을 찾아 등록한다. 코랩에서는 먼저 !apt-get -qq install fonts-noto-cjk
for path in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf"))
             + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(path)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 9, "axes.unicode_minus": False})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"
fig, ax = plt.subplots(figsize=(5.2, 3.5), dpi=260)        # 책 본문 폭(5.2인치)에 맞춘다
for seed in RUNS[16]:
    ax.plot(range(1, 17), curves[16, seed], color=BLUE, lw=1.2, alpha=1 if seed == 0 else 0.45, marker="o", ms=2.6,
            label="16에폭 학습 중 에폭마다 잰 값\n(시드 0, 연한 선은 시드 1·2)" if seed == 0 else None)
for epochs in [2, 4, 8, 16]:
    va = np.median([result[epochs, s][1] for s in RUNS[epochs]])      # 시드가 여럿이면 중앙값
    ax.scatter([epochs], [va], s=30, color=ORANGE, zorder=5)
    if epochs == 2:                                   # 2에폭 점은 점선 바로 위라 왼쪽 빈 곳에 적는다
        ax.annotate(f"2에폭\n따로 학습\n{va:.4f}", (epochs, va), xytext=(-7, 10), textcoords="offset points",
                    fontsize=8, color=ORANGE, ha="right", va="center", linespacing=1.15)
    else:
        ax.annotate(f"{epochs}에폭으로 따로 학습\n{va:.4f}", (epochs, va), xytext=(5 if epochs == 16 else 0, -7),
                    textcoords="offset points", fontsize=8, color=ORANGE, ha="right" if epochs == 16 else "center",
                    va="top", linespacing=1.15)
ax.axhline(GRAM, color=GRAY, ls="--", lw=1)
ax.text(16.6, GRAM + 0.01, f"3-그램 {GRAM:.4f}", ha="right", va="bottom", fontsize=8, color="#4b5563")
ax.set_xlabel("에폭(문장 109,995개를 몇 번 보았나)")
ax.set_ylabel("검증 손실(토큰 하나당, 자연로그)")
ax.set_xticks([1, 2, 4, 6, 8, 10, 12, 14, 16]); ax.set_xlim(-0.2, 16.8); ax.set_ylim(2.74, 3.30)
ax.grid(axis="y", color="#e5e7eb", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc="upper right")
fig.tight_layout(); fig.savefig("fig_1_1.png"); plt.close(fig)

In [ ]:
steps = math.ceil(len(X_un) / 64)                    # 1에폭의 걸음 수(배치 64)
def rate(step, epochs, lr=0.03):
    """fit()과 같은 일정: 처음 10% 걸음 동안 올리고, 코사인으로 0까지 내린다."""
    total = epochs * steps
    return lr * min(1, step / max(1, total // 10)) * 0.5 * (1 + math.cos(math.pi * step / total))
peak = max(rate(x, 16) for x in range(16 * steps))            # 워밍업에도 코사인을 곱하므로 0.03보다 조금 낮다
print(steps, f"{peak:.4f}", f"{rate(2 * steps, 2):.5f}",
      f"{rate(2 * steps, 16) / 0.03:.3f}", f"{rate(14 * steps, 16) / 0.03:.3f}")   # 설정값 0.03으로 나눈 비율

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.0), dpi=260)
for epochs, color, name in [(16, BLUE, "16에폭 학습"), (2, ORANGE, "2에폭 학습")]:
    xs = np.arange(0, epochs * steps + 1, 20)
    ax.plot(xs / steps, [rate(x, epochs) for x in xs], color=color, lw=1.8, label=name)
ax.axvline(2, color=GRAY, lw=0.8, ls=":")
y16 = rate(2 * steps, 16)
ax.scatter([2, 2], [y16, 0], color=[BLUE, ORANGE], s=24, zorder=5)
ax.annotate(f"16에폭 학습의 2에폭째\n학습률은 설정값 0.03의 {y16 / 0.03 * 100:.0f}%", (2, y16), xytext=(140, -36),
            textcoords="offset points", fontsize=8.5, color=BLUE, va="top",
            arrowprops=dict(arrowstyle="-", color=BLUE, lw=0.7))
ax.annotate("2에폭 학습은 여기서\n학습률 0으로 끝난다", (2, 0), xytext=(14, 14), textcoords="offset points",
            fontsize=8.5, color="#C2560A", va="bottom", arrowprops=dict(arrowstyle="-", color=ORANGE, lw=0.7))
ax.set_xlabel("에폭(문장 109,995개를 몇 번 보았나)")
ax.set_ylabel("학습률")
ax.set_xticks([0, 1, 2, 4, 6, 8, 10, 12, 14, 16]); ax.set_xlim(0, 16.3); ax.set_ylim(-0.0015, 0.033)
ax.legend(frameon=False, fontsize=8.5, loc="upper right")
ax.grid(axis="y", color="#e5e7eb", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_1_lr.png"); plt.close(fig)

## 1.3 손실을 글자당 비트로 바꾸기

In [ ]:
example = next(t for t in texts_va if 16 <= len(t) <= 20 and t.count(" ") >= 3)
names_of = {k: ch for ch, k in vocab.items()}
names_of.update({START: "<시작>", END: "<끝>", UNKNOWN: "<모름>", PAD: ""})
show = lambda k: "␣" if names_of[k] == " " else names_of[k]           # 공백은 ␣로 보인다
lm16 = kept[16]
lm16.eval()

def walk_of(text):
    """위치마다 (보고 있는 토큰, 정답, 정답 확률 p, -ln p, 1~3순위 후보)."""
    x = encode([text])
    with torch.no_grad():
        # 위치마다 다음 토큰 1,424개의 확률
        probs = torch.softmax(lm16(x[:, :-1]), dim=-1)[0]
    out = []
    for t in range(len(text[:64]) + 1):          # 글자마다 + <끝> 한 번
        goal = x[0, t + 1].item()
        p = probs[t, goal].item()
        top = torch.topk(probs[t], 3)
        out.append((show(x[0, t].item()), show(goal), p, -math.log(p),
                    [(show(i), v) for v, i in zip(top.values.tolist(), top.indices.tolist())]))
    return out

walk = walk_of(example)
print(example)
for k, (now, goal, p, nat, top) in enumerate(walk):
    print(f"{k + 1:>2} {align(now, 6)}→ {align(goal, 6)} p={p:.3f}  -ln p={nat:.3f}  1~3순위 "
          + ", ".join(f"{c}:{v:.2f}" for c, v in top))
mean = np.mean([w[3] for w in walk])
print(f"토큰 {len(walk)}개의 평균 {mean:.4f}나트 = {mean / math.log(2):.4f}비트, 1순위가 정답인 곳 "
      f"{sum(w[1] == w[4][0][0] for w in walk)}개")

In [ ]:
H = 4.8                                              # 그림 높이(인치). 가로세로 1단위 = 1인치로 그린다
fig, ax = plt.subplots(figsize=(5.2, H), dpi=260)
fig.subplots_adjust(0, 0, 1, 1)
n, row, first = len(walk), 0.205, H - 0.62            # 줄 수, 줄 간격, 첫 줄의 높이
X = {"pos": 0.2, "now": 0.62, "goal": 1.08, "bar": 1.38, "nat": 3.3, "top": 3.44}     # 칸의 위치(인치)
BAR = 1.2                                             # p = 1일 때 막대 길이
for key, head, ha in [("pos", "위치", "center"), ("now", "보고 있는\n토큰", "center"), ("goal", "정답", "center"),
                      ("bar", "정답에 준 확률 p", "left"), ("nat", "−ln p\n(나트)", "right"), ("top", "모델의 1~3순위 후보", "left")]:
    ax.text(X[key], H - 0.25, head, fontsize=8, fontweight="bold", color="#374151", ha=ha, va="center", linespacing=1.1)
for k, (now, goal, p, nat, top) in enumerate(walk):
    y = first - k * row
    hit = goal == top[0][0]
    ax.text(X["pos"], y, f"{k + 1}", fontsize=8, ha="center", va="center", color="#6b7280")
    ax.text(X["now"], y, now, fontsize=8.6, ha="center", va="center", color="#1f2937")
    ax.text(X["goal"], y, goal, fontsize=8.6, ha="center", va="center", color=BLUE, fontweight="bold")
    ax.add_patch(plt.Rectangle((X["bar"], y - 0.07), BAR, 0.14, color="#eef2f7", lw=0))
    ax.add_patch(plt.Rectangle((X["bar"], y - 0.07), BAR * p, 0.14, color=GREEN if hit else ORANGE, lw=0))
    ax.text(X["bar"] + BAR * p + 0.04, y, f"{p:.2f}", fontsize=8, va="center", color="#374151")
    ax.text(X["nat"], y, f"{nat:.2f}", fontsize=8, ha="right", va="center", color=RED if nat > 2.3 else "#1f2937")
    ax.text(X["top"], y, "  ".join(f"{c} {v:.2f}" for c, v in top), fontsize=8, va="center", color="#4b5563")
last = first - (n - 1) * row
ax.plot([0.05, 5.15], [H - 0.47, H - 0.47], color="#cbd5e1", lw=0.8)
ax.plot([0.05, 5.15], [last - 0.14, last - 0.14], color="#cbd5e1", lw=0.8)
ax.text(0.08, last - 0.36, f"평균 −ln p = {mean:.3f}나트 = {mean / math.log(2):.3f}비트   ·   퍼플렉서티 e^{mean:.3f} = {math.exp(mean):.1f}",
        fontsize=8.6, color=PURPLE, fontweight="bold", va="center")
ax.text(0.08, last - 0.58, "초록: 1순위가 정답   주황: 1순위가 정답이 아님", fontsize=8, color="#6b7280", va="center")
ax.set_xlim(0, 5.2); ax.set_ylim(0, H); ax.axis("off")
fig.savefig("fig_1_next.png"); plt.close(fig)

In [ ]:
ids = X_va[:3]
with torch.no_grad():
    logits = lm16(ids[:, :-1]).numpy()               # (3, 65, 1424)
goal = ids[:, 1:].numpy()
shifted = logits - logits.max(axis=-1, keepdims=True)              # 넘치지 않게 빼 둔다
log_p = shifted - np.log(np.exp(shifted).sum(axis=-1, keepdims=True))
real = goal != PAD
picked = np.take_along_axis(log_p, goal[..., None], axis=-1)[..., 0]
by_hand = -picked[real].mean()
by_torch = F.cross_entropy(torch.tensor(logits).reshape(-1, V + 2), ids[:, 1:].reshape(-1),
                           ignore_index=PAD).item()
print(int(real.sum()), f"{by_hand:.6f} {by_torch:.6f}")
print(f"{by_hand:.4f}나트 = {by_hand / math.log(2):.4f}비트, 퍼플렉서티 {math.exp(by_hand):.2f} = 2^{by_hand / math.log(2):.4f}")

In [ ]:
# 같은 문장을 어절 단위로 예측하는 모델. 모르는 어절은 글자로 풀어 적는 값을 치른다.
clean = lambda t: t[:64] == " ".join(t[:64].split()) and t[:64].strip() != ""
same = [t for t in texts_va if clean(t)]              # 공백이 하나씩인 문장만 쓴다
chars = sum(len(t[:64]) + 1 for t in same)            # 글자 수 + <끝> 하나

word_count = Counter(w for t in texts_un for w in t[:64].split())
known = {w for w, n in word_count.items() if n >= 2}
pairs, firsts = defaultdict(Counter), Counter()
for t in texts_un:
    line = ["<시작>"] + [w if w in known else "<모름>" for w in t[:64].split()] + ["<끝>"]
    for a, b in zip(line, line[1:]):
        pairs[a][b] += 1
        firsts[b] += 1
spell = Counter(ch for w, n in word_count.items() if w not in known for ch in w + "|")
va_chars = {ch for t in texts_va for ch in t}
spelled, all_words = sum(spell.values()), sum(firsts.values())

def word_loss(text):
    """어절 2-그램(0.7)과 1-그램(0.3)을 섞는다. 모르는 어절은 글자 1-그램으로 철자를 적는다."""
    line = ["<시작>"] + text[:64].split() + ["<끝>"]
    total, n = 0.0, 0
    for a, b in zip(line, line[1:]):
        a = a if a in known or a == "<시작>" else "<모름>"
        key = b if b in known or b == "<끝>" else "<모름>"
        p2 = pairs[a][key] / sum(pairs[a].values()) if pairs[a] else 0.0
        p1 = (firsts[key] + 1) / (all_words + len(known) + 2)
        total -= math.log(0.7 * p2 + 0.3 * p1)
        if key == "<모름>":                  # 어떤 글자로 된 어절인지까지 맞힌다
            total -= sum(math.log((spell[ch] + 1) / (spelled + V)) for ch in b + "|")
        n += 1
    return total, n

w_total, w_tokens = map(sum, zip(*(word_loss(t) for t in same)))
g_total, g_tokens = map(sum, zip(*(gram.sentence_loss(t) for t in same)))
X_same = encode(same)
t_mean, t_tokens = lm_score(lm16, X_same), int((X_same[:, 1:] != PAD).sum())
rows = [("어절 단위(2-그램)", w_total / w_tokens, w_tokens, w_total),
        ("글자 단위(3-그램)", g_total / g_tokens, g_tokens, g_total),
        ("글자 단위(트랜스포머, 16에폭)", t_mean, t_tokens, t_mean * t_tokens)]
va_words = [w for t in same for w in t[:64].split()]
print(len(same), chars, len(known), len(va_words), round(np.mean([w not in known for w in va_words]) * 100, 1),
      len(spell), V, len(set(spell) | va_chars | {"|"}))
print(align("모델", 30) + "토큰 수   손실   퍼플렉서티  글자당 비트")
for name, loss, n, total in rows:
    print(align(name, 30) + f"{n:>7,}  {loss:.4f} {math.exp(loss):>9.1f}   {total / math.log(2) / chars:.4f}")

In [ ]:
def word_pieces(text):
    """word_loss와 같은 계산을 어절마다 나눠 돌려준다: (어절, 나트, 모르는 어절인가)."""
    line = ["<시작>"] + text[:64].split() + ["<끝>"]
    out = []
    for a, b in zip(line, line[1:]):
        a = a if a in known or a == "<시작>" else "<모름>"
        key = b if b in known or b == "<끝>" else "<모름>"
        p2 = pairs[a][key] / sum(pairs[a].values()) if pairs[a] else 0.0
        p1 = (firsts[key] + 1) / (all_words + len(known) + 2)
        nat = -math.log(0.7 * p2 + 0.3 * p1)
        if key == "<모름>":
            nat -= sum(math.log((spell[ch] + 1) / (spelled + V)) for ch in b + "|")
        out.append((b, nat, key == "<모름>"))
    return out

other = next(t for t in same if 16 <= len(t) <= 24 and any(w not in known for w in t.split()))
compare = {}
for text in [example, other]:
    assert text in same
    w_parts = word_pieces(text)
    assert abs(sum(n for _, n, _ in w_parts) - word_loss(text)[0]) < 1e-9   # word_loss와 같은 값
    c_parts = [(goal, nat, False) for _, goal, _, nat, _ in walk_of(text)]
    compare[text] = {"어절 2-그램": w_parts, "트랜스포머(글자)": c_parts}
    print(text)
    for name, parts in compare[text].items():
        bits = sum(n for _, n, _ in parts) / math.log(2)
        chars_here = len(text[:64]) + 1
        print(f"  {align(name, 18)} 토큰 {len(parts):>2}개, 합 {bits:5.1f}비트, 글자당 {bits / chars_here:.3f}비트")
    print("  어절별 비트", [(w, round(n / math.log(2), 1), "모름" if unk else "") for w, n, unk in w_parts])

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.6), dpi=260)
bar_rows, y = [], 0
for i, (text, models) in enumerate(compare.items()):
    for name, parts in models.items():
        bar_rows.append((f"예문 {'①②'[i]} {name}", [(t, n / math.log(2), unk) for t, n, unk in parts], len(text[:64]) + 1, y))
        y -= 1.2
    y -= 0.4
top_bits = max(sum(b for _, b, _ in parts) for _, parts, _, _ in bar_rows)
ax.set_xlim(0, top_bits * 1.02); ax.set_ylim(y + 0.8, 0.75)
ax.set_yticks([]); ax.spines[["top", "right", "left"]].set_visible(False)
ax.set_xlabel("문장을 적는 데 든 비트(토큰 하나가 막대 한 칸, 왼쪽부터 순서대로)", fontsize=8.5)
ax.tick_params(axis="x", labelsize=8.5)
ax.set_title("진하고 옅은 파랑 칸이 토큰 하나씩이다. 빨강은 어휘에 없어 철자를 적은 어절", fontsize=8.2, color="#4b5563", loc="left")
fig.tight_layout()                                    # 칸의 화소 폭을 재려면 배치를 먼저 정한다
paper = fig.canvas.get_renderer()
for label, parts, chars_here, y in bar_rows:
    name = ax.text(0, y + 0.34, label, fontsize=8.4, color="#1f2937", va="bottom")
    x = 0
    for k, (tok, b, unk) in enumerate(parts):
        color = RED if unk else (BLUE if k % 2 == 0 else "#86ACEF")
        ax.barh(y, b, left=x, height=0.56, color=color, edgecolor="white", lw=0.8)
        mark = ax.text(x + b / 2, y, tok.replace(" ", "␣"), ha="center", va="center", color="white", fontweight="bold", fontsize=8)
        room = ax.transData.transform((x + b, y))[0] - ax.transData.transform((x, y))[0]
        if mark.get_window_extent(paper).width > room - 5:      # 칸에 들어가지 않는 글자는 적지 않는다
            mark.remove()
        x += b
    after = ax.transData.inverted().transform((name.get_window_extent(paper).x1 + 14, 0))[0]     # 막대 이름 바로 뒤
    ax.text(after, y + 0.34, f"{x:.1f}비트 ÷ {chars_here}글자 = {x / chars_here:.2f}", va="bottom", fontsize=8.4, color=PURPLE,
            fontweight="bold")
fig.savefig("fig_1_bits.png"); plt.close(fig)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 4.1), dpi=260)
names = ["어절 단위\n2-그램", "글자 단위\n3-그램", "글자 단위\n트랜스포머"]
ppl = [math.exp(r[1]) for r in rows]
bpc = [r[3] / math.log(2) / chars for r in rows]
for ax, values, title, fmt in [(axes[0], ppl, "① 토큰 하나당 퍼플렉서티(가로축은 로그 눈금)", "{:,.1f}"),
                               (axes[1], bpc, "② 글자 하나당 비트(같은 문장을 몇 비트로 적었나)", "{:.3f}")]:
    bars = ax.barh(names, values, color=[GRAY, GRAY, BLUE], height=0.56)
    for bar, v in zip(bars, values):
        ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2, " " + fmt.format(v),
                va="center", fontsize=8.4, color=BLUE, fontweight="bold")
    ax.set_title(title, loc="left", fontsize=8.6, color=BLUE)
    ax.invert_yaxis(); ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=8.5)
axes[0].set_xscale("log"); axes[0].set_xlim(1, 1e9)
axes[0].set_xticks([1, 10, 100, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8])
axes[0].set_xticklabels(["1", "10", "100", "1천", "1만", "10만", "100만", "1천만", "1억"])
axes[1].set_xlim(0, 6.4)
axes[0].set_xlabel("퍼플렉서티(토큰 하나당)", fontsize=8.5); axes[1].set_xlabel("글자당 비트", fontsize=8.5)
fig.tight_layout(); fig.savefig("fig_1_2.png"); plt.close(fig)

In [ ]:
# 3-그램은 1장의 글자 어휘에 없는 글자도 글자 그대로 맞혀야 한다. 그 몫을 따로 센다.
oov, oov_nat, all_nat, n_tok = 0, 0.0, 0.0, 0
for t in same:
    line = pieces([t])[0]
    for k in range(1, len(line)):
        before = line[k - 2] if k >= 2 else "<시작>"
        p1 = (gram.one[line[k]] + 1) / (gram.everything + V + 2)
        c2, c3 = gram.two.get(line[k - 1]), gram.three.get((before, line[k - 1]))
        p2 = c2[line[k]] / gram.sums2[line[k - 1]] if c2 else 0.0
        p3 = c3[line[k]] / gram.sums3[before, line[k - 1]] if c3 else 0.0
        nat = -math.log(gram.mix[0] * p1 + gram.mix[1] * p2 + gram.mix[2] * p3)   # sentence_loss와 같은 식
        all_nat += nat; n_tok += 1
        if line[k] not in vocab and line[k] != "<끝>":
            oov += 1; oov_nat += nat
print(n_tok, oov, f"{oov / n_tok * 100:.3f}%", f"{all_nat / n_tok:.4f}", f"{oov_nat / n_tok:.4f}",
      f"{(all_nat - oov_nat) / n_tok:.4f}")
kinds = set(spell) | va_chars | {"|"}
print(len(kinds), f"{sum(spell[ch] + 1 for ch in kinds) / (spelled + V):.4f}")   # 철자 확률을 모든 글자에서 더한 값

## 1.4 검증 문장에 섞인 학습 문장

In [ ]:
seen = set(texts_un)
twin = np.array([t in seen for t in texts_va])
tokens = (X_va[:, 1:] != PAD).sum(dim=1).numpy()
print(int(twin.sum()), round(twin.mean() * 100, 2), int(tokens[twin].sum()), int(tokens.sum()),
      round(tokens[twin].sum() / tokens.sum() * 100, 2), round(tokens[twin].mean(), 1), round(tokens[~twin].mean(), 1))
print(Counter(t for t in texts_va if t in seen).most_common(5))

def split_score(score_of):
    """검증 문장을 학습 자료에 있는 것과 없는 것으로 나눠 잰다."""
    return score_of(np.where(twin)[0]), score_of(np.where(~twin)[0]), score_of(np.arange(len(texts_va)))

print(align("모델", 22) + "있는 문장  없는 문장  전체")
for name, net in [("트랜스포머, 2에폭", kept[2]), ("트랜스포머, 16에폭", lm16)]:
    a, b, c = split_score(lambda rows: lm_score(net, X_va[rows]))
    print(align(name, 22) + f"{a:.4f}    {b:.4f}    {c:.4f}")
a, b, c = split_score(lambda rows: gram.score([texts_va[k] for k in rows]))
print(align("3-그램", 22) + f"{a:.4f}    {b:.4f}    {c:.4f}")

## 1.5 손실이 낮은 모델이 만든 문장

In [ ]:
char_of = {k: ch for ch, k in vocab.items()}
char_of.update({START: "", END: "", UNKNOWN: "?", PAD: ""})

def sample(net, n, seed, heat=1.0, length=64):
    """<시작>에서 출발해 확률대로 글자를 뽑는다. <끝>이 나오거나 64글자가 되면 멈춘다."""
    gen = torch.Generator().manual_seed(seed)
    ids = torch.full((n, 1), START, dtype=torch.long)
    done = torch.zeros(n, dtype=torch.bool)
    net.eval()
    with torch.no_grad():
        for _ in range(length):
            last = net(ids)[:, -1] / heat
            last[:, [PAD, START]] = float("-inf")   # 정답에 없던 토큰은 막는다
            nxt = torch.multinomial(torch.softmax(last, dim=-1), 1, generator=gen)[:, 0]
            nxt[done] = PAD
            ids = torch.cat([ids, nxt[:, None]], dim=1)
            done |= nxt == END
            if done.all():
                break
    return ["".join(char_of[k] for k in row[1:].tolist()) for row in ids]

seen64 = {t[:64] for t in texts_un}
print(align("모델", 22) + "문장  같은 문장(%)  서로 다른(%)  평균 길이")
made = {}
for name, net in [("트랜스포머, 2에폭", kept[2]), ("트랜스포머, 16에폭", lm16)]:
    texts = sample(net, 1000, seed=0)
    made[name] = texts
    copy = np.mean([t in seen64 for t in texts]) * 100
    distinct = len(set(texts)) / len(texts) * 100
    print(align(name, 22) + f"{len(texts):>4}  {copy:>10.1f}  {distinct:>11.1f}  {np.mean([len(t) for t in texts]):>8.1f}")
human = [t[:64] for t in texts_va[:1000]]
made["검증 문장(사람)"] = human
print(align("검증 문장(사람)", 22) + f"{len(human):>4}  {np.mean([t in seen64 for t in human]) * 100:>10.1f}"
      f"  {len(set(human)) / len(human) * 100:>11.1f}  {np.mean([len(t) for t in human]):>8.1f}")

In [ ]:
import random
print(align("", 22) + "10글자 이상  그 가운데 같은 문장(%)")
for name, texts in made.items():
    long_ones = [t for t in texts if len(t) >= 10]
    print(align(name, 22) + f"{len(long_ones):>9}  {np.mean([t in seen64 for t in long_ones]) * 100:>14.1f}")
print(align("", 22) + "8글자 이하(%)  64글자 이상(%)")
for name, texts in made.items():
    print(align(name, 22) + f"{np.mean([len(t) <= 8 for t in texts]) * 100:>11.1f}  {np.mean([len(t) >= 64 for t in texts]) * 100:>12.1f}")
for name in list(made)[:2]:
    print(name, Counter(made[name]).most_common(4))
    pick = random.Random(1).sample([t for t in made[name] if len(t) >= 10], 6)
    print(pick)
print({w: sum(t == w for t in texts_un) for w in ["굿", "good", "굳", "재밌다"]})   # 이 말 하나로만 된 학습 문장의 수

In [ ]:
groups = [("트랜스포머\n2에폭", kept[2]), ("트랜스포머\n16에폭", lm16), ("3-그램", None)]
vals = []
for name, net in groups:
    if net is None:
        vals.append(split_score(lambda rows: gram.score([texts_va[k] for k in rows])))
    else:
        vals.append(split_score(lambda rows: lm_score(net, X_va[rows])))
fig, ax = plt.subplots(figsize=(5.2, 3.5), dpi=260)
y = np.arange(len(groups))
for k, (label, color) in enumerate([("학습 자료에도 있는 검증 문장 311개", ORANGE),
                                     ("학습 자료에 없는 검증 문장 9,689개", BLUE),
                                     ("검증 문장 전체", GRAY)]):
    bars = ax.barh(y + (k - 1) * 0.26, [v[k] for v in vals], height=0.24, color=color, label=label)
    for bar, v in zip(bars, [v[k] for v in vals]):
        ax.text(bar.get_width() + 0.03, bar.get_y() + bar.get_height() / 2, f"{v:.4f}", va="center", fontsize=8,
                color="#4b5563" if color == GRAY else color)
ax.set_yticks(y); ax.set_yticklabels([g[0] for g in groups], fontsize=8.5); ax.invert_yaxis()
ax.set_xlim(0, max(max(v) for v in vals) * 1.16); ax.set_xlabel("검증 손실(토큰 하나당, 자연로그)", fontsize=8.5)
ax.tick_params(axis="x", labelsize=8.5)
ax.legend(frameon=False, fontsize=8, loc="lower center", bbox_to_anchor=(0.42, 1.0), ncol=2, columnspacing=1.2, handlelength=1.5)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_1_3.png"); plt.close(fig)

In [ ]:
edges = np.arange(0, 72, 4)                    # 마지막 칸은 64~67글자(64글자에서 잘린 문장)
share, average = {}, {}
for name in ["검증 문장(사람)", "트랜스포머, 2에폭", "트랜스포머, 16에폭"]:
    lengths = [len(t) for t in made[name]]
    share[name] = np.histogram(lengths, bins=edges)[0] / len(lengths) * 100     # 칸마다 문장의 비율(%)
    average[name] = np.mean(lengths)
fig, ax = plt.subplots(figsize=(5.2, 3.0), dpi=260)
for name, color, style in [("검증 문장(사람)", GRAY, "-"), ("트랜스포머, 2에폭", GREEN, "--"), ("트랜스포머, 16에폭", BLUE, "-")]:
    ax.step(edges, np.append(share[name], share[name][-1]), where="post", color=color, ls=style, lw=1.4,
            label=f"{name}: 평균 {average[name]:.1f}글자")
ax.set_xlabel("문장의 길이(글자 수, 4글자씩 묶음, 64글자에서 자름)", fontsize=8.5)
ax.set_ylabel("문장의 비율(%)", fontsize=8.5)
ax.tick_params(labelsize=8.5)
ax.set_xlim(0, 68); ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(0.27, 1.02))
ax.annotate("64글자에서\n멈춘 문장", (65, 13.6), xytext=(-12, -28), textcoords="offset points", fontsize=8, color="#4b5563",
            ha="right", va="top", arrowprops=dict(arrowstyle="-", color="#9aa1ab", lw=0.6))
ax.grid(axis="y", color="#e5e7eb", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_1_4.png"); plt.close(fig)